# Tratamento: ZS_FUT e BRL_BRL
Tratamento de dados agrícolas com arredondamento monetário preciso.

In [ ]:
import pandas as pd
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP
print('✓ Bibliotecas carregadas.')

In [ ]:
BASE_DIR = Path('.')
ARQUIVO_ZS = BASE_DIR / 'ZS_FUT.csv'
ARQUIVO_BRL = BASE_DIR / 'BRL_BRL.csv'
PASTA_SAIDA = BASE_DIR / 'indicador_agro_tratado'
PASTA_SAIDA.mkdir(exist_ok=True)
print('Caminhos configurados.')

In [ ]:
def arredondar_moeda(valor, casas=4):
    if pd.isna(valor):
        return valor
    d = Decimal(str(valor))
    q = '0.' + '0' * casas
    return float(d.quantize(Decimal(q), rounding=ROUND_HALF_UP))
print('✓ Função definida.')

## Parte 1: ZS_FUT

In [ ]:
df_zs = pd.read_csv(ARQUIVO_ZS, skiprows=2, names=['Data', 'Close', 'High', 'Low', 'Open', 'Volume'])
df_zs['Data'] = pd.to_datetime(df_zs['Data'], errors='coerce')
df_zs = df_zs.dropna(subset=['Data']).set_index('Data').sort_index()
print(f'ZS_FUT lido: {df_zs.shape}')

In [ ]:
df_zs[['Close', 'High', 'Low', 'Open']] = df_zs[['Close', 'High', 'Low', 'Open']] / 100
for col in ['Close', 'High', 'Low', 'Open']:
    df_zs[col] = df_zs[col].apply(lambda x: arredondar_moeda(x))
df_zs.columns = df_zs.columns.str.lower()
df_zs.index.name = 'Data'
print(f'ZS_FUT processado: {df_zs.shape}')

In [ ]:
df_zs.to_csv(PASTA_SAIDA / 'Soja_Chicago_USD_Bushel.csv')
print(f'✓ ZS_FUT salvo: {(PASTA_SAIDA / "Soja_Chicago_USD_Bushel.csv").stat().st_size} bytes')

## Parte 2: BRL_BRL

In [ ]:
df_brl = pd.read_csv(ARQUIVO_BRL, skiprows=2, names=['Data', 'Close', 'High', 'Low', 'Open', 'Volume'])
df_brl['Data'] = pd.to_datetime(df_brl['Data'], errors='coerce')
df_brl = df_brl.dropna(subset=['Data']).set_index('Data').sort_index()
df_brl = df_brl[['Close', 'High', 'Low', 'Open']]
df_brl = df_brl.dropna()
print(f'BRL_BRL lido: {df_brl.shape}')

In [ ]:
for col in ['Close', 'High', 'Low', 'Open']:
    df_brl[col] = df_brl[col].apply(lambda x: arredondar_moeda(x))
df_brl.columns = df_brl.columns.str.lower()
df_brl.index.name = 'Data'
print(f'BRL_BRL processado: {df_brl.shape}')

In [ ]:
df_brl.to_csv(PASTA_SAIDA / 'Cambio_USD_BRL.csv')
print(f'✓ BRL_BRL salvo: {(PASTA_SAIDA / "Cambio_USD_BRL.csv").stat().st_size} bytes')

## Resumo

In [ ]:
print('='*60)
print(f'ZS_FUT: {len(df_zs)} linhas, colunas {list(df_zs.columns)}')
print(f'BRL_BRL: {len(df_brl)} linhas, colunas {list(df_brl.columns)}')
print(f'Salvos em: {PASTA_SAIDA}')
print('='*60)